In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px


In [12]:
base_dir = Path.cwd()
data_path = (base_dir/'data'/'world_happiness_2023.csv')
df = pd.read_csv(data_path)

In [13]:
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

In [14]:
region_avg = df.groupby('Region')['Happiness_Score'].mean().reset_index().sort_values('Happiness_Score')

fig = px.bar(
    data_frame=region_avg,
    x='Happiness_Score', y='Region', orientation='h',
    color='Happiness_Score', color_continuous_scale='Blues', 
    range_color=[2.5, 7.5],
    text=region_avg['Happiness_Score'].round(2), 
    title='Western Europe leads global happiness by a wide margin, South Asia trails by over 3 points',
    labels={'Happiness_Score': '', 'Region': ''}
)

fig.update_traces(
    texttemplate='%{text:.2f}', 
    textposition='outside', 
    marker_line_width=2 
)

fig.update_layout(
    yaxis=dict(showticklabels=False, showgrid=False),
    xaxis=dict(gridcolor='lightgrey', showgrid=False),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    coloraxis_showscale=False,
    height=500
)
fig.show()

In [15]:

top8 = df.nlargest(8,'Happiness_Score').assign(Group='Top 8')
bottom8 = df.nsmallest(8,'Happiness_Score').assign(Group='Bottom 8')
combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score').reset_index(drop=True)
global_avg = df['Happiness_Score'].mean()

colors = ['#2E75B6' if g == 'Top 8' else '#E63946' for g in combined['Group']]



fig = px.bar(
    data_frame=combined,
    x='Happiness_Score', y='Country',
    orientation='h',
    color='Group',
    color_discrete_map={'Top 8': '#2E75B6', 'Bottom 8': '#E63946'},
    hover_data={'Region': True, 'Happiness_Score': ':.2f', 'Group': False},
    custom_data=combined
)


fig.update_traces(
    hovertemplate='<b>%{y}</b><br>Happiness Score: %{x:.2f}<br>'
)


fig.add_vline(x=global_avg, line_dash='dash', line_color='#888888', line_width=1.5,
              annotation_text=f'Global avg: {global_avg:.2f}',
              annotation_position='top', annotation_font=dict(size=11, color='#888888'))

gap = top8['Happiness_Score'].max() - bottom8['Happiness_Score'].min()
fig.add_annotation(
    x=6.8, y=4,
    text=f'<b>Gap: {gap:.1f} points</b><br>separates the happiest<br>and least happy nations',
    showarrow=False, font=dict(size=11, color='#333333', family='Arial'),
    bgcolor='#FFFDE7', bordercolor='#CCCCCC', borderwidth=1, borderpad=5
)
fig.add_annotation(x=8.1, y=15, text='<b>Top 8</b>',
                   showarrow=False, font=dict(color='#2E75B6', size=12))
fig.add_annotation(x=4.7, y=7, text='<b>Bottom 8</b>',
                   showarrow=False, font=dict(color='#E63946', size=12))

fig.update_layout(
    title=f'The happiest and least happy nations are {gap:.1f} points apart — happiness inequality is vast',
    xaxis=dict(range=[0, 8.8], gridcolor='#EEEEEE', title='Happiness Score (0–10)'),
    yaxis=dict(gridcolor='white', title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
    margin=dict(l=30, r=120, t=55, b=40),
    height=550
)
fig.show()
